# 3_4 — Object-held-out CV с доверительными интервалами (P0-a)

**Primary:** balanced **repeated** stratified grouped K-fold по объектам. Объекты распределяются с гарантией **CRR-объекта в каждом тест-фолде** и val из ≥2 объектов **обоих классов** (исправлены замечания о слабой стратификации/валидации). **Secondary:** LOO по объектам.

Неопределённость DPI-Flow на инференсе **пропагируется через conditional flow** (MC, `config.mc_samples_eval` сэмплов θ), а не берётся posterior mean.

> ⚠️ Сначала ПЕРЕматериализуйте `data/demo_run` (ноутбуки 1_x, `prefix_strict_preonset=True`) и переобучите модели, иначе headline-числа невалидны (утечка префикса) и устаревший `config.json`.

Корректные CI считаются в 3_5 (**object-cluster bootstrap по pooled OOF**), не наивным √n_folds.

In [1]:
import os, sys, json, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nick/Desktop/projects/liquefaction-ai


In [2]:
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.cross_validation import build_folds, evaluate_fold, aggregate_cv
import torch
device = torch.device('cpu')
QUICK = False        # True = demo-эпохи (дымовой тест)
N_SPLITS = 5
N_REPEATS = 3        # repeated CV (≥3) — требование рецензента; >1 усредняет повторы
pop, config = load_population_artifact('data/demo_run')
meta = pop['meta']
# idempotency-штамп протокола (перезапуск перезаписывает, дублей нет)
meta_stamp = {'prefix_strict_preonset': getattr(config,'prefix_strict_preonset',None),
              'seed': config.seed, 'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
              'mc_samples_eval': getattr(config,'mc_samples_eval',1), 'ts': time.strftime('%Y-%m-%d %H:%M')}
json.dump(meta_stamp, open(os.path.join(TABLES,'cv_grouped_run_meta.json'),'w'), ensure_ascii=False, indent=2)
print('samples', len(meta), '| objects', meta['object'].nunique(), '|', meta_stamp)

samples 1093 | objects 20 | {'prefix_strict_preonset': True, 'seed': 42, 'n_splits': 5, 'n_repeats': 3, 'mc_samples_eval': 8, 'ts': '2026-06-28 21:58'}


## Primary: balanced repeated grouped CV (все сильные baselines)

In [3]:
folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS, n_repeats=N_REPEATS)
raw, samples = [], []
for gid, fold in enumerate(folds):
    r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK)
    raw.append(r); samples.append(s)
    print(f"rep{fold['repeat']} fold{fold['fold']}: ", dict(zip(r.model, r.P3_Core.round(1))))
raw = pd.concat(raw, ignore_index=True); samples = pd.concat(samples, ignore_index=True)
raw.to_csv(os.path.join(TABLES,'cv_grouped_raw.csv'), index=False)        # overwrite (идемпотентно)
samples.to_csv(os.path.join(TABLES,'cv_grouped_samples.csv'), index=False)
summary = aggregate_cv(raw); summary.to_csv(os.path.join(TABLES,'cv_grouped_summary.csv'), index=False)
print('models in CV:', sorted(raw.model.unique()))
summary[[c for c in summary.columns if c in ('model','n_folds') or c.endswith('_mean')][:12]]

rep0 fold0:  {'DPI-Flow': 159.1, 'DPI-EVT': 198.5, 'EVT-NeuralSSM': 198.9, 'PINN': 100.0, 'Transformer': 136.0, 'Neural Spline Flow': 107.7, 'GRU': 40.7, 'TCN': 81.8, 'CatBoost': nan}
rep0 fold1:  {'DPI-Flow': 328.1, 'DPI-EVT': 429.4, 'EVT-NeuralSSM': 408.3, 'PINN': 100.0, 'Transformer': 235.0, 'Neural Spline Flow': 164.4, 'GRU': 102.7, 'TCN': 101.8, 'CatBoost': nan}
rep0 fold2:  {'DPI-Flow': 192.0, 'DPI-EVT': 287.2, 'EVT-NeuralSSM': 273.6, 'PINN': 100.0, 'Transformer': 205.2, 'Neural Spline Flow': 154.5, 'GRU': 175.5, 'TCN': 81.5, 'CatBoost': nan}
rep0 fold3:  {'DPI-Flow': 646.2, 'DPI-EVT': 151.7, 'EVT-NeuralSSM': 553.4, 'PINN': 100.0, 'Transformer': 375.5, 'Neural Spline Flow': 261.5, 'GRU': 73.6, 'TCN': 80.4, 'CatBoost': nan}
rep0 fold4:  {'DPI-Flow': 263.8, 'DPI-EVT': 214.4, 'EVT-NeuralSSM': 79.3, 'PINN': 100.0, 'Transformer': 178.5, 'Neural Spline Flow': 119.2, 'GRU': 79.2, 'TCN': 93.5, 'CatBoost': nan}
rep1 fold0:  {'DPI-Flow': 349.7, 'DPI-EVT': 318.5, 'EVT-NeuralSSM': 304.6, 'PI

,model,n_folds,P3_Core_mean,N_liq_logMAE_mean,N_liq_logMAE_liq_mean,Traj_RMSE_mean,Traj_RMSE_continuation_mean,Traj_RMSE_continuation_balanced_mean,Traj_RMSE_continuation_worst_mean,Traj_RMSE_worst_mean,AUROC_mean,AUPRC_mean
0,DPI-Flow,15,454.7040,0.2445,0.3221,0.0989,0.0811,0.0807,0.1110,0.1402,0.9851,0.9859
1,DPI-EVT,15,434.0389,0.2235,0.2855,0.1115,0.0958,0.0953,0.1296,0.1495,0.9823,0.9836
2,EVT-NeuralSSM,15,412.4355,0.3083,0.3299,0.1460,0.1302,0.1220,0.1875,0.2109,0.9815,0.9853
3,Transformer,15,253.8072,0.5278,0.6597,0.0483,0.0511,0.0613,0.0965,0.0791,0.9566,0.9641
4,Neural Spline Flow,15,184.3897,0.7528,0.9296,0.1148,0.1073,0.0954,0.1356,0.1418,0.9815,0.9852
5,GRU,15,111.9289,1.2665,1.4704,0.1234,0.1190,0.1286,0.1886,0.1917,0.9291,0.9444
6,PINN,15,100.0000,1.1925,1.4801,0.1592,0.1541,0.1678,0.2400,0.2766,0.9421,0.9510
7,TCN,15,87.9094,1.4353,1.7090,0.1635,0.1659,0.1849,0.2749,0.2503,0.9204,0.9363
8,CatBoost,0,NaN,0.2143,0.2454,NaN,NaN,NaN,NaN,NaN,0.9934,0.9960


> Наивный `*_ci95` в summary — только справочная дисперсия по фолдам. **Публикационные CI** — в 3_5 (object-cluster bootstrap).

## Secondary: leave-one-object-out (20 фолдов, тяжелее)

In [4]:
RUN_LOO = True
if RUN_LOO:
    lf = build_folds(meta, config, seed=42, loo=True)
    rl, sl = [], []
    for gid, fold in enumerate(lf):
        r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK); rl.append(r); sl.append(s)
    pd.concat(rl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_raw.csv'), index=False)
    pd.concat(sl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_samples.csv'), index=False)
    print('LOO done')
else: print('LOO пропущен')

LOO done
